# Phantom → 模拟 MRI 采集信号（V10.10 最小 Demo）

这个 Notebook **只执行正向仿真**，不包含重建、模型训练、优化或可视化工具库：

`动态圆形 Phantom → 材料属性图 (ρ / FF / NMIDB / dDB / T2* / f0) → water+9-fat-peaks 与 B0 → 黄金角径向多回波轨迹 → Type-3 NUDFT → 复数 k-space`

- 与原工程保持一致的采样组织：每个运动帧 **4 条径向线 × 2 个 TR × 6 个 TE**。
- 默认使用 V10.10 `echo_time`：按每个回波的实际时间对 **GT 运动坐标锚点线性插值**，在变形位置重新求 B0 和信号。该插值只涉及仿真真值，不涉及运动重建 INR。
- **依赖**：`numpy`、`matplotlib`；`finufft` 可选（有则加速，否则使用纯 NumPy 直接 NUDFT）。在本 Notebook 所在目录运行；`lib/` 与 `test.ipynb` 同级。
- 为使 CPU 运行快速，使用较小 Demo 尺寸；修改配置即可增加帧数、空间分辨率和 readout 数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from lib.phantom import PhantomConfig, SignalConfig, build_phantom
from lib.sampling import AcquisitionConfig, build_trajectory
from lib.acquisition import simulate_acquisition

np.set_printoptions(precision=3, suppress=True)

## 1. 创建动态 Phantom + 材料信号参数

这里 `gt[y, x, f, :] = [x(t), y(t), M0]`，坐标是归一化 FOV；`maps` 是绑定在材料点上的静态参数。原工程 720 帧的周期长度和 `n=128` 均保留为库中的默认设置；此处缩减为 12 帧的可运行示例。

In [ ]:
ph_cfg = PhantomConfig(
    n=96,
    cycle_lengths=(6, 6),        # Demo: 两个运动周期，共 12 个 GT 锚点帧
    outer_radius_px=(18., 28., 38., 46.),
)
sig_cfg = SignalConfig()      # 与原工程一致的 9-fat-peak、B0、T2* 等参数

gt, maps, meta = build_phantom(ph_cfg, sig_cfg)
selected_frames = np.array([0, 2], dtype=np.int32)

print('GT [y, x, frame, (x,y,M0)]:', gt.shape)
print('active material points:', meta['active_mask'].sum())
print('signal maps:', list(maps))

fig, ax = plt.subplots(1, 4, figsize=(13, 3))
ax[0].imshow(maps['rho'], cmap='gray'); ax[0].set_title('ρ (material M0)')
ax[1].imshow(maps['ff'], cmap='viridis'); ax[1].set_title('Fat fraction')
ax[2].imshow(maps['b0_hz'], cmap='coolwarm'); ax[2].set_title('B0 [Hz], canonical')
disp = np.linalg.norm(gt[:, :, selected_frames[-1], :2] - meta['canonical_xy'], axis=-1)*ph_cfg.n
ax[3].imshow(disp, cmap='magma'); ax[3].set_title('Motion [pixels], frame 2')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 2. 生成黄金角径向多回波轨迹

每帧 4 条线，包含反复使用的导航线（索引 0）；其余 3 条按黄金角推进。每条线有 2 次 TR，每个 TR 6 个 echo。`tr_start_time_ms`、`echo_time_ms` 是**绝对采集时间**，`te_ms` 是 TR 内的相对回波时间。

In [ ]:
acq_cfg = AcquisitionConfig(
    n=ph_cfg.n,
    n_readout=32,
    n_motion_frames=ph_cfg.n_frames,
    tr_ms=15.,
)
trajectory = build_trajectory(acq_cfg)
print('trajectory [frame, line, readout, xy]:', trajectory['ktraj_frames'].shape)
print('TE per TR [ms]:', trajectory['te_ms'])
print('example echo timestamps [ms]:', trajectory['echo_time_ms'][0, 0, 0])

plt.figure(figsize=(4, 4))
for kline in trajectory['ktraj_frames'][0]:
    plt.plot(kline[:, 0], kline[:, 1], '.', markersize=3)
plt.gca().set_aspect('equal'); plt.title('4 radial lines (frame 0)')
plt.xlabel('kx [cycles/FOV]'); plt.ylabel('ky [cycles/FOV]')
plt.tight_layout(); plt.show()

## 3. 正向模拟采集，得到复数 k-space

- `motion_time_sampling='echo_time'`：逐 echo 运动时间（V10.10 默认）。可改成 `'tr_start'`（逐 TR）或 `'frame_held'`（旧版本整帧不动）。
- `backend='auto'`：若安装 `finufft` 则使用 FINUFFT；否则由 NumPy 直接计算 `Σ_p S_p(TE) exp(-i 2π k·x_p(t))`。本 Demo 不要求 PyTorch。
- 返回数组的第一个维度只包含 `selected_frames`，不是全部 GT 帧。

In [ ]:
kspace = simulate_acquisition(
    gt, maps, trajectory,
    frame_indices=selected_frames,
    motion_time_sampling='echo_time',
    motion_frame_duration_ms=acq_cfg.motion_frame_acquisition_ms,
    b0_peak_hz=sig_cfg.b0_peak_hz,
    larmor_hz_per_ppm=sig_cfg.larmor_hz_per_ppm,
    backend='auto',   # 无 finufft 也可运行
)
assert kspace.shape == (len(selected_frames), 4, 2, 6, acq_cfg.n_readout)
assert kspace.dtype == np.complex64 and np.isfinite(kspace).all()

print('kspace shape [frame, line, TR, echo, readout]:', kspace.shape)
print('kspace dtype:', kspace.dtype)
print('sample complex signal:', kspace[1, 0, 0, 2, acq_cfg.n_readout//2])

## 4. 查看信号并导出结果

左图显示一帧中 4 条径向线的回波幅度；右图为导航线中心 k-space 点随 12 个 TE 的幅度。输出文件 `simulated_signal.npz` 会生成在 Notebook 运行目录中；**它不是压缩包的预装内容**。

In [ ]:
center_k = acq_cfg.n_readout // 2
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].imshow(np.log1p(np.abs(kspace[1, :, 0, 2, :])), aspect='auto', cmap='magma')
ax[0].set(xlabel='readout point', ylabel='radial line', title='log(1+|kspace|): frame 2, TR0, echo2')
ax[1].plot(trajectory['te_ms'].reshape(-1),
           np.abs(kspace[1, 0, :, :, center_k]).reshape(-1), 'o-')
ax[1].set(xlabel='TE [ms]', ylabel='|kspace(center)|', title='Navigator center: 12 echoes')
plt.tight_layout(); plt.show()

output_npz = 'simulated_signal.npz'
np.savez_compressed(
    output_npz,
    kspace=kspace,
    frame_indices=selected_frames,
    ktraj_frames=trajectory['ktraj_frames'][selected_frames],
    te_ms=trajectory['te_ms'],
    tr_start_time_ms=trajectory['tr_start_time_ms'][selected_frames],
    echo_time_ms=trajectory['echo_time_ms'][selected_frames],
)
print(f'Saved {output_npz} ({kspace.shape}, {kspace.dtype})')